In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

In [3]:
from langgraph.graph.message import add_messages

class Chatstate(TypedDict):
    
    messages: Annotated[list[BaseMessage], add_messages]

In [6]:
llm = ChatOpenAI()

def chat_node(state: ChatState):
    
    # take user query from state
    messages = state['messages']
    
    # send to llm
    response = llm.invoke(messages)
    
    # response store state
    return {'messages' : [response]}

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

#add node
graph.add_node('chat_node', chat_node)


#add edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer = checkpointer)


NameError: name 'ChatState' is not defined

In [ ]:
initial_state = {
    'messages' : [HumanMessage(content = 'What is the capital of India')]
}
chatbot.invoke(initial_state)['messages'][-1].content

In [ ]:
thread_id = '1'

while True:
    user_message = input('Type here: ')
    
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break
    
    config = {'configurable': ('thread_id': thread_id)}
    response = chatbot.invoke({'messages' : [HumanMessage(content = user_message)]}, config = config)
    
    print('AI: ', response['messages'][-1].content)

In [ ]:
chatbot.get_state(config = config)